# Bronze Ingestion
Read all tables from the MirroredDB mirrored database and write them as Delta tables in the bronze layer.

In [ ]:
# Configuration
mirrored_db_name = "MirroredDB"
workspace_id = "27a7b3cb-0878-4d2b-9dd6-182ca3feb47e"
bronze_path = "Tables/bronze"

In [ ]:
# Read all table names from the mirrored database using three-part naming
tables_df = spark.sql(f"SHOW TABLES IN `{mirrored_db_name}`")
table_names = [row.tableName for row in tables_df.collect()]
print(f"Found {len(table_names)} tables in {mirrored_db_name}: {table_names}")

In [ ]:
# Ingest each table into bronze layer as Delta tables
from pyspark.sql.functions import current_timestamp, lit
import datetime

batch_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

for table_name in table_names:
    print(f"Ingesting {table_name}...")
    df = spark.sql(f"SELECT * FROM `{mirrored_db_name}`.`{table_name}`")
    
    # Add metadata columns
    df = df.withColumn("_ingestion_timestamp", current_timestamp()) \
           .withColumn("_batch_id", lit(batch_id)) \
           .withColumn("_source_table", lit(table_name))
    
    # Write as Delta table in bronze (overwrite mode)
    df.write.format("delta").mode("overwrite").saveAsTable(f"bronze_{table_name}")
    print(f"  -> bronze_{table_name}: {df.count()} rows")

print(f"\nBronze ingestion complete. {len(table_names)} tables ingested.")

In [ ]:
# Verify bronze tables
bronze_tables = [t for t in spark.catalog.listTables() if t.name.startswith("bronze_")]
print(f"Bronze tables created: {len(bronze_tables)}")
for t in bronze_tables:
    count = spark.table(t.name).count()
    print(f"  {t.name}: {count} rows")